<a href="https://colab.research.google.com/github/cccontre1/AI-Medical-Transcriptor/blob/main/ReporterIA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎙️ Laboratorio Clínico: ReporterIA Local (PoC)
**Objetivo:** Desarrollar una Prueba de Concepto (PoC) para la transcripción y estructuración de informes radiológicos utilizando modelos de IA 100% locales (Open Source).

**Justificación Clínica:** La atención de urgencias y el diagnóstico radiológico requieren inmediatez y privacidad absoluta. Reemplazar APIs externas por un flujo local garantiza la protección de los datos del paciente y disminuye el riesgo de alucinaciones al utilizar modelos calibrados para el dominio médico.

**Arquitectura del Pipeline:**
1. **Audio a Texto:** `Faster-Whisper` (con filtro de silencios VAD).
2. **Texto a Estructura:** `MedGemma` (vía Ollama).
3. **Control de Salida:** Forzado de formato JSON estricto para asegurar el calce perfecto con las plantillas clínicas.

## ⚙️ Módulo 1: Infraestructura y Entorno (GPU)
En esta sección preparamos el servidor de Google Colab simulando las condiciones de un servidor de producción.
* **Verificación de Hardware:** El sistema comprobará automáticamente si la aceleración por Tarjeta Gráfica (GPU) está activa para garantizar tiempos de respuesta clínicos.
* **Dependencias:** Instalamos las herramientas de compresión y librerías base (`ollama`, `faster-whisper`, `gradio`).
* **Motor Local:** Levantamos el motor de inferencia en segundo plano y descargamos los pesos del modelo *MedGemma*.

In [ ]:
import os
import sys

# ==========================================
# 0. Verificación de Seguridad (Requisito de GPU)
# ==========================================
print("🔍 Verificando hardware...")

# El comando 'nvidia-smi' solo responde si hay una GPU activa
if os.system("nvidia-smi > /dev/null 2>&1") != 0:
    print("\n" + "🔴"*30)
    print("❌ ERROR CRÍTICO: No se detectó una Tarjeta Gráfica (GPU).")
    print("🔴"*30)
    print("\nPara que la transcripción y el modelo funcionen en segundos, debes activar la GPU.")
    print("Por favor, sigue estos 3 pasos rápidos en el menú de arriba:")
    print("  1. Haz clic en 'Entorno de ejecución'.")
    print("  2. Selecciona 'Cambiar tipo de entorno de ejecución'.")
    print("  3. En 'Acelerador de hardware', elige 'T4 GPU' y haz clic en Guardar.")
    print("\n👉 Una vez hecho eso, vuelve a darle 'Play' a esta celda.")
    print("="*60 + "\n")
    sys.exit("Ejecución detenida intencionalmente para evitar sobrecarga de CPU.")
else:
    print("✅ GPU detectada correctamente. Procediendo con la instalación...\n")

# ==========================================
# 1. Instalación de Infraestructura
# ==========================================
!sudo apt-get update -qq && sudo apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh
!pip install -q ollama gradio faster-whisper nest-asyncio

# ==========================================
# 2. Arranque y Descarga de Modelos
# ==========================================
!nohup ollama serve > ollama.log 2>&1 &
!sleep 5
print("\nIniciando descarga de MedGemma (esto tomará unos minutos)...")
!ollama pull medgemma
print("\n¡Entorno listo y modelo descargado con éxito!")

🔍 Verificando hardware...
✅ GPU detectada correctamente. Procediendo con la instalación...

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
zstd is already the newest version (1.4.8+dfsg-3build1).
0 upgraded, 0 newly installed, 0 to remove and 88 not upgraded.
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.

Iniciando descarga de MedGemma (esto tomará unos mi

## 🗂️ Módulo 2: Configuración Clínica y Plantillas Propias (El futuro `config.py`)
Para mantener el código ordenado y adaptado a la realidad médica, aislamos todas las variables estáticas aquí. Esto incluye:
* **Diccionario de Plantillas Propias:** Un repositorio estructurado (`MIS_PLANTILLAS`) donde se ingresan y gestionan los formatos exactos de los diferentes exámenes, garantizando que la salida de la IA respete las estructuras validadas institucionalmente.
* **Prompt del Sistema (Reglas Generales):** Instrucciones inquebrantables para el modelo (ej. no inventar hallazgos, respetar la lateralidad, estructurar en JSON).
* **Parámetros del Modelo:** Temperatura (configurada en 0.0 para forzar el máximo determinismo, cero creatividad y evitar alucinaciones clínicas).

In [ ]:
# ==========================================
# Módulo 2: Configuración y Reglas Clínicas
# ==========================================

# 1. Variables de Inferencia
MODELO_LLM = "medgemma"
TEMPERATURA_LLM = 0.0

# 2. Tus Plantillas Propias (Diccionario de Plantillas)
# Aquí puedes agregar tantas plantillas como necesites.
# Solo ponles un nombre clave en mayúsculas.
MIS_PLANTILLAS = {
    "RX_TORAX_ESTANDAR": """ESTUDIO: Radiografía de Tórax (Frente y Perfil)
TÉCNICA: [X]
HALLAZGOS:
- Vía aérea y pleura: [X]
- Parénquima pulmonar: [X]
- Silueta cardiovascular: [X]
- Estructuras óseas: [X]
IMPRESIÓN DIAGNÓSTICA: [X]""",

    "ECO_ABDOMINAL_NORMAL": """ESTUDIO: Ecografía Abdominal
TÉCNICA: Exploración transabdominal con transductor convexo.
HALLAZGOS:
- Hígado: [X]
- Vesícula y vía biliar: [X]
- Páncreas: [X]
- Bazo: [X]
- Riñones: [X]
IMPRESIÓN DIAGNÓSTICA: [X]"""
}

# 3. Reglas Estrictas del Sistema (System Prompt)
REGLAS_SISTEMA = """Eres un asistente médico experto en radiología. Tu única tarea es extraer información de la transcripción de un dictado médico y estructurarla usando la plantilla proporcionada.

REGLAS INQUEBRANTABLES:
1. RESPONDE ÚNICA Y EXCLUSIVAMENTE CON UN OBJETO JSON VÁLIDO.
2. NO INVENTES DATOS. Si un hallazgo no se menciona, escribe: "Sin alteraciones evidentes en el dictado."
3. RESPETA LA LATERALIDAD estrictamente.

ESTRUCTURA DEL JSON REQUERIDA:
{
    "metadata_paciente": {
        "edad": "",
        "sexo": ""
    },
    "hallazgos_positivos": [],
    "informe_final": "Aquí va el texto completo de la plantilla seleccionada con la información inyectada, manteniendo los saltos de línea (\n).",
    "advertencias": []
}"""

# Por defecto para las primeras pruebas usaremos la primera,
# pero luego en la interfaz web (Módulo 6) haremos que el usuario elija cuál usar.
PLANTILLA_ACTIVA = MIS_PLANTILLAS["RX_TORAX_ESTANDAR"]

print("✅ Módulo 2 (Configuración y Reglas) cargado en memoria con plantillas personalizadas.")

✅ Módulo 2 (Configuración y Reglas) cargado en memoria con plantillas personalizadas.


## 🎧 Módulo 3: Motor de Procesamiento de Audio (`audio_pipeline`)
Este módulo se encarga exclusivamente de la capa acústica utilizando `faster-whisper`, una implementación optimizada que corre excepcionalmente rápido en la GPU.

**Optimizaciones Clínicas Clave:**
1. **Precisión de hardware (`float16`):** Reduce el consumo de memoria de la tarjeta gráfica a la mitad sin perder precisión en el reconocimiento de palabras médicas.
2. **Filtro de Silencios (VAD):** Los dictados médicos suelen tener pausas mientras se revisan las imágenes. El parámetro `vad_filter=True` detecta y elimina los silencios prolongados antes de transcribir, evitando que el modelo "invente" ruido de fondo o palabras fantasma.

In [ ]:
import time
from faster_whisper import WhisperModel

# ==========================================
# Módulo 3: Motor de Transcripción de Audio
# ==========================================

print("⏳ Cargando modelo acústico (Whisper) en la memoria de la GPU...")
# Usamos el modelo "base" que es ultrarrápido. Si en el futuro necesitas más precisión
# con términos muy raros, puedes cambiar "base" por "small" o "medium".
audio_model = WhisperModel("medium", device="cuda", compute_type="float16")
print("✅ Motor de audio cargado exitosamente.")

def procesar_dictado_audio(ruta_audio: str) -> str:
    """
    Toma un archivo de audio (soporta .ogg de WhatsApp, .wav, .mp3, etc.)
    y devuelve la transcripción limpia en texto.
    """
    if not ruta_audio:
        raise ValueError("No se recibió ningún archivo de audio.")

    print(f"🎙️ Procesando audio: {ruta_audio} ...")
    inicio_ms = time.time()

    # beam_size=5 mejora la precisión matemática de la predicción de palabras
    # vad_filter=True es nuestro escudo contra los silencios largos
    segmentos, info = audio_model.transcribe(
        ruta_audio,
        language="es",
        beam_size=5,
        vad_filter=True,
        vad_parameters=dict(min_silence_duration_ms=500)
    ) # <--- El error ocurría porque faltaba este paréntesis de cierre

    # Whisper devuelve un "generador" (procesa a medida que lee).
    # Aquí unimos todos los fragmentos en un solo texto.
    texto_transcrito = " ".join([segmento.text for segmento in segmentos])

    fin_ms = time.time()
    tiempo_total = round(fin_ms - inicio_ms, 2)
    print(f"⏱️ Transcripción completada en {tiempo_total} segundos.")

    return texto_transcrito.strip()



⏳ Cargando modelo acústico (Whisper) en la memoria de la GPU...
✅ Motor de audio cargado exitosamente.


## 🧠 Módulo 4: Extracción y Estructuración Clínica (`clinical_llm`)
Este módulo recibe la transcripción cruda y se comunica con `MedGemma` a través del servidor local de Ollama.

**Optimizaciones Clínicas Clave:**
1. **Ensamblaje del Contexto:** Toma el dictado y lo envuelve con las Reglas del Sistema y la Plantilla específica seleccionada por el usuario.
2. **Control de Salida (Structured Output):** Utiliza el parámetro `format='json'` para garantizar algorítmicamente que el modelo no genere texto conversacional, devolviendo únicamente un objeto de datos procesable.
3. **Parseo Seguro:** Implementa expresiones regulares (Regex) para limpiar la respuesta en caso de que el modelo devuelva el JSON envuelto en bloques de código Markdown.

In [ ]:
import json
import re
import time
import ollama

# ==========================================
# Módulo 4: Motor de Estructuración Clínica
# ==========================================

def extraer_json_seguro(texto_crudo: str) -> dict:
    """
    Limpia la respuesta del LLM. A veces los modelos devuelven el JSON
    envuelto en ```json ... ```. Esta función extrae solo el diccionario puro.
    """
    try:
        # Busca cualquier cosa que parezca un objeto JSON { ... }
        match = re.search(r'\{.*\}', texto_crudo, re.DOTALL)
        if match:
            return json.loads(match.group(0))
        # Si no tiene envoltura, lo intenta procesar directo
        return json.loads(texto_crudo)
    except Exception as e:
        print(f"⚠️ Error al parsear JSON: {e}")
        # Devuelve un diccionario de error estructurado para que el sistema no colapse
        return {
            "error": "El modelo no devolvió un JSON válido.",
            "texto_crudo": texto_crudo
        }

def generar_informe_clinico(transcripcion: str, clave_plantilla: str) -> dict:
    """
    Envía el dictado a MedGemma y lo fuerza a devolver un JSON basado en la plantilla.
    """
    if not transcripcion:
        return {"error": "No hay transcripción para procesar."}

    print("🧠 MedGemma está analizando el caso...")
    inicio_ms = time.time()

    # 1. Recuperamos la plantilla elegida del diccionario del Módulo 2
    plantilla_texto = MIS_PLANTILLAS.get(clave_plantilla, MIS_PLANTILLAS["RX_TORAX_ESTANDAR"])

    # 2. Ensamblamos el Prompt del Usuario (lo que realmente lee el modelo)
    prompt_usuario = f"""
    PLANTILLA A UTILIZAR:
    {plantilla_texto}

    DICTADO MÉDICO (TRANSCRIPCIÓN):
    {transcripcion}
    """

    # 3. Llamada local a Ollama
    try:
        respuesta = ollama.chat(
            model=MODELO_LLM,
            messages=[
                {"role": "system", "content": REGLAS_SISTEMA},
                {"role": "user", "content": prompt_usuario}
            ],
            format='json', # ¡Esta es la clave para la automatización!
            options={"temperature": TEMPERATURA_LLM}
        )

        texto_respuesta = respuesta['message']['content']
        datos_json = extraer_json_seguro(texto_respuesta)

    except Exception as e:
        datos_json = {"error": f"Fallo en la comunicación con Ollama: {str(e)}"}

    fin_ms = time.time()
    print(f"⏱️ Análisis clínico completado en {round(fin_ms - inicio_ms, 2)} segundos.")

    return datos_json

# ==========================================
# Prueba Rápida del Módulo (Opcional)
# ==========================================
# dictado_prueba = "Paciente masculino de 45 años. Radiografía de tórax. Vemos un infiltrado en la base pulmonar derecha. No hay derrame. Silueta cardíaca normal."
# resultado = generar_informe_clinico(dictado_prueba, "RX_TORAX_ESTANDAR")
# print(json.dumps(resultado, indent=2, ensure_ascii=False))

## 🔗 Módulo 5: Ensamblaje y Control de Calidad (`main.py`)
Aquí orquestamos el flujo completo de la Prueba de Concepto. Esta función central emula la capa de lógica de negocio y trazabilidad del sistema.

**Responsabilidades del Orquestador:**
1. **Llamada Secuencial:** Ejecuta la transcripción (Módulo 3) y la estructuración (Módulo 4) en cadena.
2. **Manejo de Errores:** Intercepta audios vacíos o fallas de inferencia para evitar que el sistema colapse.
3. **Capa de Trazabilidad:** Mide el tiempo total de procesamiento y empaqueta la transcripción cruda junto con el JSON estructurado, asegurando que el médico auditor pueda comparar lo que se dictó vs. lo que la IA estructuró.

In [ ]:
import time

# ==========================================
# Módulo 5: Orquestador Principal
# ==========================================

def procesar_caso_completo(ruta_audio: str, clave_plantilla: str) -> dict:
    """
    Función principal que une Whisper + MedGemma.
    Simula el flujo de producción y empaqueta la respuesta con metadata de trazabilidad.
    """
    print(f"\n🚀 Iniciando procesamiento completo usando plantilla: {clave_plantilla}")
    inicio_total = time.time()

    # 1. Transcripción de Audio
    try:
        texto_transcrito = procesar_dictado_audio(ruta_audio)
    except Exception as e:
        return {"ok": False, "error": f"Fallo en el pipeline de audio: {str(e)}"}

    if not texto_transcrito:
        return {"ok": False, "error": "El audio fue procesado pero se detectó como puro silencio."}

    # 2. Estructuración Clínica con LLM
    resultado_llm = generar_informe_clinico(texto_transcrito, clave_plantilla)

    # 3. Ensamblaje y Control de Calidad
    if "error" in resultado_llm:
        return {"ok": False, "transcripcion": texto_transcrito, "error": resultado_llm["error"]}

    # Armamos el paquete final estandarizado
    paquete_final = {
        "ok": True,
        "metodo": "local_poc_v1",
        "tiempo_procesamiento_seg": round(time.time() - inicio_total, 2),
        "transcripcion": texto_transcrito,
        "metadata_paciente": resultado_llm.get("metadata_paciente", {}),
        "hallazgos_positivos": resultado_llm.get("hallazgos_positivos", []),
        "informe_final": resultado_llm.get("informe_final", "Error: Informe vacío."),
        "advertencias": resultado_llm.get("advertencias", [])
    }

    # Validación extra: si el informe final perdió el formato, lanzamos una alerta interna
    if "\n" not in paquete_final.get("informe_final", ""):
        paquete_final["advertencias"].append("Alerta de Formato: El modelo omitió los saltos de línea estructurales.")

    print(f"✅ Caso orquestado exitosamente en {paquete_final['tiempo_procesamiento_seg']} segundos.")
    return paquete_final

print("✅ Módulo 5 (Orquestador) listo en memoria.")

✅ Módulo 5 (Orquestador) listo en memoria.


## 🖥️ Módulo 6: Interfaz de Validación Clínica (Gradio)
Esta interfaz gráfica permite a los profesionales médicos interactuar con la Prueba de Concepto de manera intuitiva.

**Características de la Interfaz:**
* **Entrada de Audio Dual:** Permite grabación en tiempo real mediante micrófono o la carga de archivos locales (soportando formatos de notas de voz).
* **Selector Dinámico:** Un menú desplegable que lee automáticamente las plantillas configuradas en el Módulo 2 (`MIS_PLANTILLAS`).
* **Visualización de Trazabilidad:** Muestra de forma transparente el texto crudo entendido por Whisper, el informe final listo para firma y el JSON técnico con métricas de tiempo para auditoría.

In [ ]:
import os
import time
print("🔌 Encendiendo el servidor de Ollama...")
os.system("nohup ollama serve > ollama.log 2>&1 &")
time.sleep(5)
print("✅ ¡Ollama está despierto y listo!")

🔌 Encendiendo el servidor de Ollama...
✅ ¡Ollama está despierto y listo!


In [ ]:
import gradio as gr
import nest_asyncio

# ==========================================
# Módulo 6: Interfaz Gráfica (Gradio) - ReporterIA
# ==========================================

# 🛡️ Vacuna contra el error de subida de archivos en Colab
nest_asyncio.apply()
gr.close_all()

# Extraemos las claves de las plantillas que definiste en el Módulo 2
opciones_plantillas = list(MIS_PLANTILLAS.keys())

def adaptador_gradio(ruta_audio, clave_plantilla):
    """
    Esta función hace de puente entre la página web de Gradio y nuestro Módulo 5.
    Maneja el caso en que el usuario presione el botón sin haber subido audio.
    """
    if not ruta_audio:
        return "⚠️ Error: Por favor graba o sube un audio primero.", "Esperando dictado...", {}

    resultado = procesar_caso_completo(ruta_audio, clave_plantilla)

    if not resultado.get("ok", False):
        error_msg = resultado.get("error", "Error desconocido en el procesamiento.")
        return error_msg, "Fallo en la generación del informe.", resultado

    return resultado["transcripcion"], resultado["informe_final"], resultado

# Construcción visual de la página bajo la nueva identidad: ReporterIA
with gr.Blocks(title="ReporterIA - Experimento Local", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🎙️ ReporterIA V4 - Entorno de Validación Clínica")
    gr.Markdown("**Arquitectura 100% Local:** Modelo Acústico (`faster-whisper` Medium) + Extracción Clínica (`MedGemma` vía Ollama)")

    with gr.Row():
        # Columna Izquierda: Entradas del Usuario
        with gr.Column(scale=1):
            audio_in = gr.Audio(sources=["microphone", "upload"], type="filepath", label="1. Dictado Médico")
            plantilla_in = gr.Dropdown(choices=opciones_plantillas, value=opciones_plantillas[0], label="2. Seleccionar Plantilla Clínica")
            btn_generar = gr.Button("Generar Informe Estructurado", variant="primary", size="lg")

        # Columna Derecha: Resultados y Trazabilidad
        with gr.Column(scale=2):
            with gr.Accordion("Ver Transcripción Cruda (Whisper)", open=False):
                transcripcion_out = gr.Textbox(label="Texto sin procesar", show_copy_button=True, interactive=False)

            informe_out = gr.Textbox(label="Informe Final (MedGemma)", lines=12, show_copy_button=True)

            with gr.Accordion("Ver Trazabilidad y Metadata (JSON)", open=False):
                json_out = gr.JSON(label="Datos Técnicos")

    # Conexión del botón con la función
    btn_generar.click(
        fn=adaptador_gradio,
        inputs=[audio_in, plantilla_in],
        outputs=[transcripcion_out, informe_out, json_out]
    )

print("🌐 Levantando servidor web...")
# Lanzamos la interfaz. Recuerda: debug=True mantendrá la celda corriendo infinitamente (es normal)
demo.launch(share=True, debug=True)

/tmp/ipykernel_14135/853285373.py:27: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="ReporterIA - Experimento Local", theme=gr.themes.Soft()) as demo:


🌐 Levantando servidor web...
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://bc59d798b5dd772def.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/protocols/http/httptools_impl.py", line 421, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/middleware/proxy_headers.py", line 62, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/fastapi/applications.py", line 1159, in __call__
    await super().__call__(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/applications.py", line 107, in __call__
    await self.middleware_stack(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/errors.py", line 186, in __call__
    raise exc
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/error


🚀 Iniciando procesamiento completo usando plantilla: ECO_ABDOMINAL_NORMAL
🎙️ Procesando audio: /tmp/gradio/275b856a64cadef04979aa78a7f12e40d9dce69d2cd9bfca46ed4cee6f59dcc8/WhatsApp Ptt 2026-06-17 at 11.25.52 PM.ogg ...
⏱️ Transcripción completada en 2.55 segundos.
🧠 MedGemma está analizando el caso...
⏱️ Análisis clínico completado en 13.79 segundos.
✅ Caso orquestado exitosamente en 16.34 segundos.


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/protocols/http/httptools_impl.py", line 421, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/middleware/proxy_headers.py", line 62, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/fastapi/applications.py", line 1159, in __call__
    await super().__call__(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/applications.py", line 107, in __call__
    await self.middleware_stack(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/errors.py", line 186, in __call__
    raise exc
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/error


🚀 Iniciando procesamiento completo usando plantilla: ECO_ABDOMINAL_NORMAL
🎙️ Procesando audio: /tmp/gradio/2b6be1a8527bd6d4e6c06041327aa5bfd4dbc0e1479e048cf5b0f950492a0f38/WhatsApp Ptt 2026-06-17 at 11.25.07 PM.ogg ...
⏱️ Transcripción completada en 2.61 segundos.
🧠 MedGemma está analizando el caso...
⏱️ Análisis clínico completado en 12.5 segundos.
✅ Caso orquestado exitosamente en 15.11 segundos.

🚀 Iniciando procesamiento completo usando plantilla: ECO_ABDOMINAL_NORMAL
🎙️ Procesando audio: /tmp/gradio/275b856a64cadef04979aa78a7f12e40d9dce69d2cd9bfca46ed4cee6f59dcc8/WhatsApp Ptt 2026-06-17 at 11.25.52 PM.ogg ...
⏱️ Transcripción completada en 3.09 segundos.
🧠 MedGemma está analizando el caso...
⏱️ Análisis clínico completado en 13.35 segundos.
✅ Caso orquestado exitosamente en 16.43 segundos.
